Original v1 preprocessing, unchanged except for file paths. It reproduces the one-to-many join that duplicated participants; see nhanes_chd_audit_and_revalidation.ipynb.

# NHANES Heart Disease Preprocessing

This notebook selects the requested columns from the six NHANES CSV files, documents any missing matches, merges them into a single dataset, and then reproduces the preprocessing pipeline described in the target paper (missing-data handling, one-hot encoding, normalization, train/test split, and feature selection with supporting EDA).

In [1]:

from __future__ import annotations

from pathlib import Path
from difflib import SequenceMatcher
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.over_sampling import SMOTE


In [2]:

# Paths are relative to the repository root: run from `notebooks/` or the repo root
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed_v1"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

raw_files = {
    "demographic.csv": RAW_DIR / "demographic.csv",
    "diet.csv": RAW_DIR / "diet.csv",
    "examination.csv": RAW_DIR / "examination.csv",
    "labs.csv": RAW_DIR / "labs.csv",
    "medications.csv": RAW_DIR / "medications.csv",
    "questionnaire.csv": RAW_DIR / "questionnaire.csv",
}

target_columns = [
    "Gender",
    "Age",
    "Systolic",
    "Diastolic",
    "Platelet Count",
    "Red-Cell Distribution Width",
    "Cholesterol",
    "Creatinine",
    "Glucose",
    "LDH",
    "Uric Acid",
    "Triglycerides",
    "HDL",
    "Glycohemoglobin",
    "Vigorous Work",
    "Moderate Work",
    "Diabetes",
    "Blood-Rel Diabetes",
    "Blood-Rel Stroke",
]


In [3]:

# Resolve requested features to NHANES columns (manual + fuzzy fallback)
def read_column_names(path: Path) -> List[str]:
    try:
        return list(pd.read_csv(path, nrows=0).columns)
    except UnicodeDecodeError:
        return list(pd.read_csv(path, nrows=0, encoding="latin1").columns)

column_names = {name: read_column_names(path) for name, path in raw_files.items()}

manual_matches: Dict[str, Optional[Tuple[str, str]]] = {
    "Gender": ("demographic.csv", "RIAGENDR"),
    "Age": ("demographic.csv", "RIDAGEYR"),
    "Systolic": ("examination.csv", "BPXSY1"),
    "Diastolic": ("examination.csv", "BPXDI1"),
    "Platelet Count": ("labs.csv", "LBXPLTSI"),
    "Red-Cell Distribution Width": ("labs.csv", "LBXRDW"),
    "Cholesterol": ("labs.csv", "LBXSCH"),
    "Creatinine": ("labs.csv", "LBXSCR"),
    "Glucose": ("labs.csv", "LBXSGL"),
    "LDH": ("labs.csv", "LBXSLDSI"),
    "Uric Acid": ("labs.csv", "LBXSUA"),
    "Triglycerides": ("labs.csv", "LBXTR"),
    "HDL": ("labs.csv", "LBDHDD"),
    "Glycohemoglobin": ("labs.csv", "LBXGH"),
    "Vigorous Work": ("questionnaire.csv", "PAQ605"),
    "Moderate Work": ("questionnaire.csv", "PAQ620"),
    "Diabetes": ("questionnaire.csv", "DIQ010"),
    "Blood-Rel Diabetes": ("questionnaire.csv", "MCQ300C"),
    "Blood-Rel Stroke": ("questionnaire.csv", "MCQ160F"),
}


def best_match(target: str) -> Tuple[Optional[str], Optional[str], float]:
    best_file, best_col, best_score = None, None, 0.0
    for file_name, cols in column_names.items():
        for col in cols:
            score = SequenceMatcher(None, target.lower(), col.lower()).ratio()
            if score > best_score:
                best_file, best_col, best_score = file_name, col, score
    return best_file, best_col, best_score

resolved_matches: Dict[str, Dict[str, Optional[object]]] = {}
missing_targets: List[str] = []
threshold = 0.45

for target in target_columns:
    if target in manual_matches:
        file_name, column = manual_matches[target]
        available_cols = column_names.get(file_name, [])
        if column not in available_cols:
            resolved_matches[target] = {
                "file": None,
                "column": None,
                "match_score": None,
                "notes": f"{target}: manual code {column} not present in {file_name}",
            }
            missing_targets.append(target)
            continue
        resolved_matches[target] = {
            "file": file_name,
            "column": column,
            "match_score": 1.0,
            "notes": "manual mapping",
        }
        continue

    file_name, column, score = best_match(target)
    if score >= threshold:
        resolved_matches[target] = {
            "file": file_name,
            "column": column,
            "match_score": round(score, 3),
            "notes": "auto-selected",
        }
    else:
        resolved_matches[target] = {
            "file": None,
            "column": None,
            "match_score": round(score, 3),
            "notes": "no close match",
        }
        missing_targets.append(target)

available_targets = [t for t in target_columns if t not in missing_targets]

match_df = pd.DataFrame.from_dict(resolved_matches, orient="index")
match_df


,file,column,match_score,notes
Gender,demographic.csv,RIAGENDR,1.0,manual mapping
Age,demographic.csv,RIDAGEYR,1.0,manual mapping
Systolic,examination.csv,BPXSY1,1.0,manual mapping
Diastolic,examination.csv,BPXDI1,1.0,manual mapping
Platelet Count,labs.csv,LBXPLTSI,1.0,manual mapping
Red-Cell Distribution Width,labs.csv,LBXRDW,1.0,manual mapping
Cholesterol,labs.csv,LBXSCH,1.0,manual mapping
Creatinine,labs.csv,LBXSCR,1.0,manual mapping
Glucose,labs.csv,LBXSGL,1.0,manual mapping
LDH,labs.csv,LBXSLDSI,1.0,manual mapping


In [4]:

# Pull columns, merge on SEQN, engineer labels, and drop targets we cannot source
columns_by_file: dict[str, set[str]] = {name: {"SEQN"} for name in raw_files}

for target, info in resolved_matches.items():
    file_name = info["file"]
    column = info["column"]
    if file_name and column:
        columns_by_file[file_name].add(column)

columns_by_file["questionnaire.csv"].add("MCQ160C")  # CHD indicator used for the label

selected_frames: dict[str, pd.DataFrame] = {}
for file_name, cols in columns_by_file.items():
    try:
        df = pd.read_csv(raw_files[file_name], usecols=sorted(cols), low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(raw_files[file_name], usecols=sorted(cols), low_memory=False, encoding="latin1")
    selected_frames[file_name] = df

merged_df = selected_frames["demographic.csv"].copy()
for file_name, df in selected_frames.items():
    if file_name == "demographic.csv":
        continue
    merged_df = merged_df.merge(df, on="SEQN", how="outer")

rename_map = {info["column"]: target for target, info in resolved_matches.items() if info["column"] is not None}
renamed_df = merged_df.rename(columns=rename_map)

renamed_df["Gender"] = renamed_df["Gender"].map({1: "Male", 2: "Female"})

if "Blood-Rel Stroke" in renamed_df:
    renamed_df["Blood-Rel Stroke"] = renamed_df["Blood-Rel Stroke"].map({1: "Yes", 2: "No"})

renamed_df["disease_status"] = renamed_df["MCQ160C"].map({1: "diseased", 2: "normal"})

renamed_df = renamed_df.drop(columns=[c for c in ["MCQ160C"] if c in renamed_df.columns])

for col in available_targets:
    if col not in renamed_df.columns:
        renamed_df[col] = np.nan

selected_df = renamed_df[["SEQN"] + available_targets + ["disease_status"]]

selected_path = PROCESSED_DIR / "selected_features.csv"
selected_df.to_csv(selected_path, index=False)

dropped_cols = [t for t in target_columns if t not in available_targets]
print("Saved selected features to", selected_path.relative_to(PROJECT_ROOT).as_posix())
if dropped_cols:
    print("Dropped targets (missing in raw files):", dropped_cols)
else:
    print("All targets were matched.")

selected_df.head()


Saved selected features to data/processed_v1/selected_features.csv
All targets were matched.


,SEQN,Gender,Age,Systolic,Diastolic,Platelet Count,Red-Cell Distribution Width,Cholesterol,Creatinine,Glucose,...,Uric Acid,Triglycerides,HDL,Glycohemoglobin,Vigorous Work,Moderate Work,Diabetes,Blood-Rel Diabetes,Blood-Rel Stroke,disease_status
0,73557,Male,69,122.0,72.0,204.0,14.0,168.0,1.21,554.0,...,3.3,NaN,65.0,13.9,2.0,2.0,1.0,1.0,Yes,normal
1,73557,Male,69,122.0,72.0,204.0,14.0,168.0,1.21,554.0,...,3.3,NaN,65.0,13.9,2.0,2.0,1.0,1.0,Yes,normal
2,73558,Male,54,156.0,62.0,314.0,13.4,167.0,0.79,219.0,...,4.7,NaN,50.0,9.1,2.0,1.0,1.0,1.0,No,normal
3,73558,Male,54,156.0,62.0,314.0,13.4,167.0,0.79,219.0,...,4.7,NaN,50.0,9.1,2.0,1.0,1.0,1.0,No,normal
4,73558,Male,54,156.0,62.0,314.0,13.4,167.0,0.79,219.0,...,4.7,NaN,50.0,9.1,2.0,1.0,1.0,1.0,No,normal


In [5]:
# Quick EDA: correlations (numeric only) + class balance
sns.set_theme(style="whitegrid", context="notebook")

numeric_cols = selected_df.select_dtypes(include=[np.number]).columns
correlation_matrix = selected_df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, cmap="coolwarm", center=0)
plt.title("Correlation heatmap (numeric features)")
plt.tight_layout()
plt.close("all")

plt.figure(figsize=(5, 4))
sns.countplot(data=selected_df, x="disease_status", order=["diseased", "normal"], palette="Set2")
plt.title("Class balance")
plt.tight_layout()
plt.close("all")


## Workflow overview

- Select and harmonize requested NHANES features across files keyed by SEQN.
- Engineer labels (MCQ160C) and keep family history flags; drop unmapped targets.
- Run quick EDA for correlations and class balance.
- Preprocess with imputers, one-hot encoding, and z-score scaling; save the full preprocessed dataset.
- Create a stratified train/test split, then balance the training set with SMOTE; save all splits.


In [6]:

# Preprocess: impute missing values, one-hot encode categoricals, scale numerics, balance train set
feature_columns = available_targets
target_column = "disease_status"

categorical_features = [
    "Gender",
    "Vigorous Work",
    "Moderate Work",
    "Diabetes",
    "Blood-Rel Diabetes",
    "Blood-Rel Stroke",
]

categorical_features = [c for c in categorical_features if c in feature_columns]
numeric_features = sorted(set(feature_columns) - set(categorical_features))

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
)

valid_mask = selected_df[target_column].notna()
print(f"Rows with valid disease_status label: {valid_mask.sum()} / {len(selected_df)}")
df_valid = selected_df.loc[valid_mask].reset_index(drop=True)

X = df_valid[feature_columns]
y = df_valid[target_column]

X_processed = preprocessor.fit_transform(X)
if hasattr(X_processed, "toarray"):
    X_processed = X_processed.toarray()
processed_feature_names = preprocessor.get_feature_names_out()
readable_feature_names = [name.split("__", 1)[-1] if "__" in name else name for name in processed_feature_names]

X_processed_df = pd.DataFrame(X_processed, columns=readable_feature_names)

X_processed_df[target_column] = y.values

processed_path = PROCESSED_DIR / "heart_disease_preprocessed.csv"
X_processed_df.to_csv(processed_path, index=False)
print("Saved preprocessed data to", processed_path.relative_to(PROJECT_ROOT).as_posix())

X_model = X_processed_df.drop(columns=target_column)
y_model = X_processed_df[target_column]

print("Class distribution:", y_model.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.30,
    stratify=y_model,
    random_state=42,
)

train_df = X_train.copy()
train_df[target_column] = y_train.values

test_df = X_test.copy()
test_df[target_column] = y_test.values

train_path = PROCESSED_DIR / "train_split.csv"
test_path = PROCESSED_DIR / "test_split.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("Train set saved to", train_path.relative_to(PROJECT_ROOT).as_posix(), "| shape:", train_df.shape)
print("Test set saved to", test_path.relative_to(PROJECT_ROOT).as_posix(), "| shape:", test_df.shape)

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

smote_train_df = pd.DataFrame(X_train_balanced, columns=X_train.columns)
smote_train_df[target_column] = y_train_balanced

smote_train_path = PROCESSED_DIR / "train_split_smote.csv"
smote_train_df.to_csv(smote_train_path, index=False)

print("Post-SMOTE train class distribution:", y_train_balanced.value_counts())
print("SMOTE-balanced train set saved to", smote_train_path.relative_to(PROJECT_ROOT).as_posix(), "| shape:", smote_train_df.shape)


Rows with valid disease_status label: 15103 / 20194


Saved preprocessed data to data/processed_v1/heart_disease_preprocessed.csv
Class distribution: disease_status
normal      13562
diseased     1541
Name: count, dtype: int64


Train set saved to data/processed_v1/train_split.csv | shape: (10572, 32)
Test set saved to data/processed_v1/test_split.csv | shape: (4531, 32)


Post-SMOTE train class distribution: disease_status
normal      9493
diseased    9493
Name: count, dtype: int64
SMOTE-balanced train set saved to data/processed_v1/train_split_smote.csv | shape: (18986, 32)
